# 00 - Spark Và MinIO Setup

## Import Local Utilities

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "utils").exists():
    notebooks_dir = cwd
elif (cwd / "notebooks" / "utils").exists():
    notebooks_dir = cwd / "notebooks"
else:
    notebooks_dir = cwd

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from utils.spark_session import (
    BRONZE_GREEN_PATH,
    BRONZE_WEATHER_PATH,
    BRONZE_YELLOW_PATH,
    GOLD_ROOT_PATH,
    SILVER_CLEAN_PATH,
    SILVER_CORE_PATH,
    SILVER_QUALITY_PATH,
    SILVER_TAXI_WEATHER_PATH,
    SILVER_WEATHER_PATH,
    count_by_partition,
    get_spark,
    null_profile,
    path_exists,
    safe_display,
    show_schema,
)

print(f"Using notebooks utilities from: {notebooks_dir}")

Using notebooks utilities from: /tmp/metropulse-run/notebooks


## Create Spark Session

In [2]:
spark = get_spark("MetroPulse 00 Spark MinIO Setup")

print(f"Spark version: {spark.version}")
print(f"Spark timezone: {spark.conf.get('spark.sql.session.timeZone')}")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/metropulse-notebook-ivy-20260526/cache
The jars for the packages stored in: /tmp/metropulse-notebook-ivy-20260526/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-37ebaac7-6665-44fa-a388-1dad59979fd7;1.0
	confs: [default]


	found org.apache.hadoop#hadoop-aws;3.3.4 in central


	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central


	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (97ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...


	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (1713ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final/wildfly-openssl-1.0.7.Final.jar ...
	[SUCCESSFUL ] org.wildfly.openssl#wildfly-openssl;1.0.7.Final!wildfly-openssl.jar (108ms)
:: resolution report :: resolve 3688ms :: artifacts dl 1924ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   3   |   3   |   0   ||   3   |   3   |
	----------------------------------------------

	3 artifacts copied, 0 already retrieved (275421kB/170ms)


26/05/26 03:15:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.1


Spark timezone: America/New_York


## Canonical S3A Paths

Các đường dẫn bên dưới xác định phạm vi dữ liệu được kiểm chứng ở Bronze và Silver. Endpoint và credentials được lấy từ môi trường chạy, vì notebook không lưu secrets trong nội dung phân tích.

In [3]:
canonical_paths = {
    "BRONZE_YELLOW_PATH": BRONZE_YELLOW_PATH,
    "BRONZE_GREEN_PATH": BRONZE_GREEN_PATH,
    "BRONZE_WEATHER_PATH": BRONZE_WEATHER_PATH,
    "SILVER_WEATHER_PATH": SILVER_WEATHER_PATH,
    "SILVER_TAXI_WEATHER_PATH": SILVER_TAXI_WEATHER_PATH,
    "SILVER_CORE_PATH": SILVER_CORE_PATH,
    "SILVER_CLEAN_PATH": SILVER_CLEAN_PATH,
    "SILVER_QUALITY_PATH": SILVER_QUALITY_PATH,
    "GOLD_ROOT_PATH": GOLD_ROOT_PATH,
}

for name, path in canonical_paths.items():
    print(f"{name}: {path}")

BRONZE_YELLOW_PATH: s3a://bronze/yellow_taxi/
BRONZE_GREEN_PATH: s3a://bronze/green_taxi/
BRONZE_WEATHER_PATH: s3a://bronze/weather/
SILVER_WEATHER_PATH: s3a://silver/hourly_weather/
SILVER_TAXI_WEATHER_PATH: s3a://silver/taxi_weather_trips/
SILVER_CORE_PATH: s3a://silver/taxi_weather_trips_core/
SILVER_CLEAN_PATH: s3a://silver/taxi_weather_trips_clean/
SILVER_QUALITY_PATH: s3a://silver/quality_reports/silver_core_quality/latest/
GOLD_ROOT_PATH: s3a://gold/


## Path Existence Check

In [4]:
path_status_rows = []

for name, path in canonical_paths.items():
    try:
        exists = path_exists(spark, path)
        status = "available" if exists else "missing"
        error = ""
    except Exception as exc:
        exists = False
        status = "check_failed"
        error = str(exc)[:300]

    path_status_rows.append((name, path, exists, status, error))

path_status_df = spark.createDataFrame(
    path_status_rows,
    ["path_name", "path", "exists", "status", "error"],
)
safe_display(path_status_df, n=len(path_status_rows), truncate=False)

26/05/26 03:15:06 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+------------------------+--------------------------------------------------------+------+---------+-----+
|path_name               |path                                                    |exists|status   |error|
+------------------------+--------------------------------------------------------+------+---------+-----+
|BRONZE_YELLOW_PATH      |s3a://bronze/yellow_taxi/                               |true  |available|     |
|BRONZE_GREEN_PATH       |s3a://bronze/green_taxi/                                |true  |available|     |
|BRONZE_WEATHER_PATH     |s3a://bronze/weather/                                   |true  |available|     |
|SILVER_WEATHER_PATH     |s3a://silver/hourly_weather/                            |true  |available|     |
|SILVER_TAXI_WEATHER_PATH|s3a://silver/taxi_weather_trips/                        |true  |available|     |
|SILVER_CORE_PATH        |s3a://silver/taxi_weather_trips_core/                   |true  |available|     |
|SILVER_CLEAN_PATH       |s3a://silve

## Read One Available Dataset Gracefully

In [5]:
read_candidates = [
    ("bronze_yellow", BRONZE_YELLOW_PATH, "parquet"),
    ("bronze_green", BRONZE_GREEN_PATH, "parquet"),
    ("bronze_weather", BRONZE_WEATHER_PATH, "parquet"),
    ("silver_weather", SILVER_WEATHER_PATH, "parquet"),
    ("silver_clean", SILVER_CLEAN_PATH, "parquet"),
]

selected_name = None
selected_path = None
selected_df = None

for name, path, fmt in read_candidates:
    try:
        if not path_exists(spark, path):
            print(f"Skip missing path: {name} -> {path}")
            continue

        selected_df = spark.read.format(fmt).load(path)
        selected_name = name
        selected_path = path
        print(f"Selected dataset: {selected_name}")
        print(f"Selected path: {selected_path}")
        break
    except Exception as exc:
        print(f"Could not read {name} at {path}: {exc}")

if selected_df is None:
    print("No Bronze or Silver dataset is available yet. Run pipeline jobs first, then rerun this notebook.")

Selected dataset: bronze_yellow
Selected path: s3a://bronze/yellow_taxi/


## Schema And Small Preview


In [6]:
if selected_df is not None:
    show_schema(selected_df)
    safe_display(selected_df.limit(20), n=20, truncate=False)
else:
    print("Skipping schema and preview because no dataset was selected.")

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- kafka_timestamp_type: integer (nullable = true)
 |-- key: string (nullable = true)
 |-- json_data: string (nullable = true)
 |-- taxi_type: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = true)



+---------------+---------+--------+-----------------------+--------------------+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-----------------------+--------------+
|topic          |partition|offset  |kafka_timestamp        |kafka_timestamp_type|key |json_data                                                                                                                                                                               

## Helper Demonstrations

Tổng hợp theo partition và null profile minh họa cách tôi kiểm chứng dữ liệu ở mức aggregate. Các phép đo chỉ áp dụng cho những cột xuất hiện trong schema đọc được, nhờ vậy kết quả không nhầm cột không tồn tại với giá trị null bằng không.

In [7]:
partition_candidates = [
    "ingestion_date",
    "pickup_year_month",
    "weather_year_month",
    "taxi_type",
]
important_columns = [
    "topic",
    "partition",
    "offset",
    "kafka_timestamp",
    "json_data",
    "pickup_datetime",
    "pickup_hour",
    "taxi_type",
    "temperature_f",
    "precipitation_mm",
    "is_gold_candidate",
]

if selected_df is None:
    print("Skipping helper demonstrations because no dataset was selected.")
else:
    selected_columns = set(selected_df.columns)
    partition_col = next(
        (column_name for column_name in partition_candidates if column_name in selected_columns),
        None,
    )

    if partition_col:
        print(f"Row count by partition column: {partition_col}")
        safe_display(count_by_partition(selected_df, partition_col), n=50, truncate=False)
    else:
        print("No known partition column found for count_by_partition demonstration.")

    available_important_columns = [
        column_name for column_name in important_columns if column_name in selected_columns
    ]

    if available_important_columns:
        print("Null profile for available important columns:")
        safe_display(null_profile(selected_df, available_important_columns), n=50, truncate=False)
    else:
        print("No important columns found for null_profile demonstration.")

Row count by partition column: ingestion_date



[Stage 4:==>                                                      (4 + 4) / 107]




[Stage 4:=====>                                                  (11 + 4) / 107]




[Stage 4:=============>                                          (26 + 4) / 107]




[Stage 4:==================>                                     (36 + 4) / 107]



...


[Stage 10:================>                                      (33 + 4) / 107]




[Stage 10:====================>                                  (39 + 4) / 107]




[Stage 10:======================>                                (43 + 4) / 107]




[Stage 10:=========================================>             (80 + 4) / 107]



+---------------+----------+----------+
|column_name    |null_count|null_ratio|
+---------------+----------+----------+
|topic          |0         |0.0       |
|partition      |0         |0.0       |
|offset         |0         |0.0       |
|kafka_timestamp|0         |0.0       |
|json_data      |0         |0.0       |
|taxi_type      |0         |0.0       |
+---------------+----------+----------+



## Kết Luận Thiết Lập

Kết quả thiết lập xác nhận Spark có thể đọc MinIO theo cấu hình phân tích thống nhất. Trên cơ sở đó, phần Bronze tiếp theo kiểm chứng raw Kafka metadata, nội dung `json_data` và tính truy vết của dữ liệu ingestion.